# Task #45 — Kiểm tra và lưu dataset đặc trưng mới (Story #8, giai đoạn 1)

Dựng lại dataframe đặc trưng (cùng logic Task #44, `notebooks/16_build_feature_dataset.ipynb`, tự chứa lại từ đầu theo đúng quy ước các notebook trước), kiểm tra sanity check, rồi lưu thành `data/processed/orders_features.csv` — dataset đặc trưng đầu tiên tách biệt với `orders_labeled.csv` gốc.

In [1]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
df = pd.read_csv(PROCESSED_DIR / "orders_labeled.csv", low_memory=False)

date_cols = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

bool_cols = [
    "payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
    "payment_has_not_defined", "payment_has_voucher", "items_multi_seller",
]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

df["is_delayed"] = df["is_delayed"].astype("boolean")

items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
items_weight = items.merge(products[["product_id", "product_weight_g"]], on="product_id", how="left")
order_weight = items_weight.groupby("order_id")["product_weight_g"].sum(min_count=1).rename("items_total_weight_g")
df = df.merge(order_weight, left_on="order_id", right_index=True, how="left")

df["approval_gap_hours"] = (df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / 3600
df["estimated_delivery_days"] = (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
df["order_purchase_month"] = df["order_purchase_timestamp"].dt.month

df_onehot = pd.get_dummies(
    df, columns=["customer_state", "primary_seller_state"],
    prefix=["customer_state", "primary_seller_state"],
)
customer_state_cols = [c for c in df_onehot.columns if c.startswith("customer_state_")]
seller_state_cols = [c for c in df_onehot.columns if c.startswith("primary_seller_state_")]

numeric_features = [
    "items_num_items", "items_num_products", "items_num_sellers",
    "items_total_price", "items_total_freight", "items_num_categories",
    "items_total_weight_g",
    "payment_total_value", "payment_num_rows", "payment_num_types", "payment_max_installments",
    "payment_value_boleto", "payment_value_credit_card", "payment_value_debit_card",
    "payment_value_not_defined", "payment_value_voucher",
    "approval_gap_hours", "estimated_delivery_days", "order_purchase_month",
]
boolean_features = [
    "payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
    "payment_has_not_defined", "payment_has_voucher", "items_multi_seller",
]
region_onehot_features = customer_state_cols + seller_state_cols
final_columns = ["order_id"] + numeric_features + boolean_features + region_onehot_features + ["is_delayed"]

features_df = df_onehot[final_columns].copy()
print(features_df.shape)

(99441, 77)


## 1. Kiểm tra kiểu dữ liệu

In [2]:
features_df.dtypes.value_counts()

bool       50
float64    18
boolean     7
str         1
int32       1
Name: count, dtype: int64

## 2. Sanity check — one-hot vùng

Mỗi đơn phải có đúng 1 cột `customer_state_*` = 1 (khách hàng luôn có state). `primary_seller_state_*` có thể toàn 0 nếu đơn không xác định được seller chính (775 đơn thiếu items/seller, ghi nhận ở Task #28/Story #4).

In [3]:
customer_onehot_sum = features_df[customer_state_cols].sum(axis=1)
seller_onehot_sum = features_df[seller_state_cols].sum(axis=1)

print("customer_state one-hot — phân phối tổng theo dòng:")
print(customer_onehot_sum.value_counts())
print("\nprimary_seller_state one-hot — phân phối tổng theo dòng:")
print(seller_onehot_sum.value_counts())

customer_state one-hot — phân phối tổng theo dòng:
1    99441
Name: count, dtype: int64

primary_seller_state one-hot — phân phối tổng theo dòng:
1    98666
0      775
Name: count, dtype: int64


## 3. Tỉ lệ giá trị thiếu theo cột (chưa xử lý — để dành giai đoạn 2)

In [4]:
null_pct = (features_df.isna().sum() / len(features_df) * 100).round(2)
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
print(f"{len(null_pct)}/{len(features_df.columns)} cột có giá trị thiếu:")
null_pct

9/77 cột có giá trị thiếu:


is_delayed              2.98
items_total_weight_g    0.80
items_num_items         0.78
items_num_products      0.78
items_num_sellers       0.78
items_total_freight     0.78
items_total_price       0.78
items_num_categories    0.78
approval_gap_hours      0.16
dtype: float64

## 4. Phân phối nhãn `is_delayed`

Không phải đặc trưng, giữ lại để dùng ở giai đoạn 2 (chia train/test + xử lý mất cân bằng).

In [5]:
features_df["is_delayed"].value_counts(dropna=False)

is_delayed
False    88649
True      7827
<NA>      2965
Name: count, dtype: Int64

## 5. Lưu dataset đặc trưng

Lưu thành `data/processed/orders_features.csv`, tách biệt với `orders_labeled.csv` gốc (không đổi). Lưu ý cho notebook đọc lại sau: CSV không giữ dtype qua vòng đọc/ghi (gotcha đã ghi nhận ở Task #32) — phải ép kiểu lại `is_delayed`/`payment_has_*`/`items_multi_seller` thành `"boolean"` và các cột one-hot thành `bool` khi đọc lại.

In [6]:
OUT_PATH = PROCESSED_DIR / "orders_features.csv"
features_df.to_csv(OUT_PATH, index=False)
print(f"Đã lưu dataset đặc trưng: {len(features_df)} dòng, {len(features_df.columns)} cột vào {OUT_PATH}")

Đã lưu dataset đặc trưng: 99441 dòng, 77 cột vào ..\data\processed\orders_features.csv
